# 01 — Dataset Anatomy (THINGS-EEG)

Basic EDA for all 10 participants. Loads data per participant, generates grand-average ERP, per-channel ERP, and single-condition plots. Saves figures to `artifacts/figures/sub-XX/` for each participant.

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else (Path.cwd() / ".." / "..").resolve()
DATA_ROOT = ROOT / "data"
ARTIFACTS_BASE = ROOT / "artifacts" / "figures"
PARTICIPANTS = [f"sub-{i:02d}" for i in range(1, 11)]

In [10]:
def load_preprocessed(path):
    """Load preprocessed EEG .npy file. Returns (data, ch_names, times)."""
    d = np.load(path, allow_pickle=True).item()
    return d["preprocessed_eeg_data"], d["ch_names"], d["times"]

## Process all participants

For each participant: load data, print shape summary, generate and save three figures to `artifacts/figures/sub-XX/`.

In [11]:
DATE_TAG = "2026-02-25"
saved_paths = []

for sub in PARTICIPANTS:
    sub_dir = DATA_ROOT / sub
    if not (sub_dir / "preprocessed_eeg_training.npy").exists():
        print(f"Skip {sub}: no data")
        continue
    out_dir = ARTIFACTS_BASE / sub
    out_dir.mkdir(parents=True, exist_ok=True)

    X_train, ch_names, times = load_preprocessed(sub_dir / "preprocessed_eeg_training.npy")
    X_test, ch_names_te, times_te = load_preprocessed(sub_dir / "preprocessed_eeg_test.npy")
    assert ch_names == ch_names_te and np.allclose(times, times_te)

    erp_train = X_train.mean(axis=(0, 1))[np.newaxis]
    erp_test = X_test.mean(axis=(0, 1))[np.newaxis]
    X0 = X_train[0] if X_train.ndim == 5 else X_train

    print(f"{sub}: train {X_train.shape}, test {X_test.shape}")

    # 1. Grand average ERP
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for p in range(erp_train.shape[0]):
        axes[0].plot(times * 1000, erp_train[p].mean(axis=0), alpha=0.7)
    axes[0].axvline(0, color="gray", ls="--", alpha=0.7)
    axes[0].set_xlabel("Time (ms)")
    axes[0].set_ylabel("Amplitude (µV)")
    axes[0].set_title(f"{sub}: Train channel-average ERP")
    axes[0].grid(True, alpha=0.3)
    for p in range(erp_test.shape[0]):
        axes[1].plot(times * 1000, erp_test[p].mean(axis=0), alpha=0.7)
    axes[1].axvline(0, color="gray", ls="--", alpha=0.7)
    axes[1].set_xlabel("Time (ms)")
    axes[1].set_title(f"{sub}: Test channel-average ERP")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    p1 = out_dir / f"eda__grand_average_erp__{DATE_TAG}.png"
    plt.savefig(p1, dpi=200, bbox_inches="tight")
    saved_paths.append(str(p1.relative_to(ROOT)))
    plt.close()

    # 2. Per-channel ERP
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, ch in enumerate(ch_names):
        ax.plot(times * 1000, erp_train[0][i], alpha=0.7, label=ch)
    ax.axvline(0, color="gray", ls="--", alpha=0.7)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_title(f"{sub}: Per-channel ERP")
    ax.legend(ncol=3, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p2 = out_dir / f"eda__per_channel_erp__{DATE_TAG}.png"
    plt.savefig(p2, dpi=200, bbox_inches="tight")
    saved_paths.append(str(p2.relative_to(ROOT)))
    plt.close()

    # 3. Single condition (4 reps)
    cond0 = X0[0]
    ch_avg_trials = cond0.mean(axis=1)
    fig, ax = plt.subplots(figsize=(8, 4))
    for r in range(min(4, cond0.shape[0])):
        ax.plot(times * 1000, ch_avg_trials[r], alpha=0.6, label=f"rep {r+1}")
    ax.plot(times * 1000, ch_avg_trials.mean(axis=0), "k-", lw=2, label="mean")
    ax.axvline(0, color="gray", ls="--", alpha=0.7)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_title(f"{sub}: Condition 0, 4 repetitions")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p3 = out_dir / f"eda__single_condition_reps__{DATE_TAG}.png"
    plt.savefig(p3, dpi=200, bbox_inches="tight")
    saved_paths.append(str(p3.relative_to(ROOT)))
    plt.close()

print(f"\nSaved {len(saved_paths)} figures to artifacts/figures/")

sub-01: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-02: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-03: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-04: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-05: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-06: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-07: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-08: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-09: train (16540, 4, 17, 100), test (200, 80, 17, 100)
sub-10: train (16540, 4, 17, 100), test (200, 80, 17, 100)

Saved 30 figures to artifacts/figures/


## Dataset summary

Per participant: train (16540 conditions × 4 reps), test (200 conditions × 80 reps), 17 channels, 100 timepoints (-0.2 to 0.79 s).

In [17]:
# Value range sanity check for all participants; save to artifacts
rows = []
for sub in PARTICIPANTS:
    p = DATA_ROOT / sub / "preprocessed_eeg_training.npy"
    if not p.exists():
        continue
    X, _, _ = load_preprocessed(p)
    rows.append({
        "participant": sub,
        "min": X.min(),
        "max": X.max(),
        "mean": X.mean(),
        "std": X.std(),
    })

stats_df = pd.DataFrame(rows)
stats_path = ROOT / "artifacts" / "tables" / "eda__value_stats_all_participants__2026-02-25.csv"
stats_path.parent.mkdir(parents=True, exist_ok=True)
stats_df.to_csv(stats_path, index=False)
print(stats_df.to_string(index=False))
print(f"\nSaved to {stats_path.relative_to(ROOT)}")

participant         min        max      mean      std
     sub-01 -328.758442  85.146189 -0.035036 0.840087
     sub-02  -67.811845  19.348271 -0.045118 0.780827
     sub-03  -29.874819  27.810837 -0.061383 0.768140
     sub-04 -180.341181 191.111006 -0.093927 0.815328
     sub-05  -32.336822  32.339231 -0.061616 0.904777
     sub-06  -42.754898  26.406420 -0.030514 0.923096
     sub-07  -15.175645   8.453562 -0.026997 0.788252
     sub-08  -16.346178  14.886945 -0.082024 0.822352
     sub-09  -19.073668  17.860902 -0.040822 0.777033
     sub-10  -13.601058  11.266018 -0.047036 0.780902

Saved to artifacts\tables\eda__value_stats_all_participants__2026-02-25.csv


## Outputs

Figures saved to `artifacts/figures/sub-XX/` per participant (see results.md for full catalog).

In [13]:
for p in saved_paths[:6]:
    print(p)
if len(saved_paths) > 6:
    print(f"... and {len(saved_paths) - 6} more")

artifacts\figures\sub-01\eda__grand_average_erp__2026-02-25.png
artifacts\figures\sub-01\eda__per_channel_erp__2026-02-25.png
artifacts\figures\sub-01\eda__single_condition_reps__2026-02-25.png
artifacts\figures\sub-02\eda__grand_average_erp__2026-02-25.png
artifacts\figures\sub-02\eda__per_channel_erp__2026-02-25.png
artifacts\figures\sub-02\eda__single_condition_reps__2026-02-25.png
... and 24 more
